# 04 — Báo cáo chất lượng và benchmark
Chỉ sử dụng dữ liệu đã chạy qua notebook 03. Không có giá trị thời gian/số dòng SEC được điền sẵn.

In [ ]:
from pathlib import Path
import os
import sys

candidate = Path(os.environ.get("SEC_PROJECT_ROOT", Path.cwd())).resolve()
locations = (Path.cwd().resolve(), Path.cwd().resolve().parent, candidate)
code_root = next((p for p in locations if (p / "src").is_dir()), None)
if code_root is None:
    raise RuntimeError("Hãy mở notebook từ sec_spark_project hoặc thư mục notebooks của dự án")
if str(code_root) not in sys.path:
    sys.path.insert(0, str(code_root))
os.environ.setdefault("SEC_PROJECT_ROOT", str(code_root))
from src.config import make_spark, DATA, REPORTS, QUARTERS, TABLES
spark = make_spark()
print("Spark", spark.version, "|", spark.sparkContext.master, "|", QUARTERS)


In [ ]:
from src.report import report_all, SQL_QUERIES
summary, nulls, timings = report_all(spark)
summary

In [ ]:
nulls.sort_values(["table", "column", "stage"])

## Biểu đồ null trước/sau
Null thật có thể tăng sau khi chuẩn hóa rỗng hoặc đánh dấu parse lỗi. Biểu đồ missing gồm chuỗi trắng cho phép so sánh nhất quán hơn.

In [ ]:
from IPython.display import display, Image
for table in TABLES:
    display(Image(filename=str(REPORTS / f"{table}_null_pct.png")))
    display(Image(filename=str(REPORTS / f"{table}_missing_including_blank_pct.png")))

## Benchmark TXT/Parquet
Đã warm-up, lặp 3 lần đổi thứ tự và lấy median. Đo scan 3 cột có checksum, không đo lazy read đơn thuần. Cache hệ điều hành không được kiểm soát; không suy rộng hệ số tốc độ sang máy khác.

In [ ]:
import pandas as pd
pd.DataFrame(timings)[["quarter", "format", "rows", "median_seconds", "bytes_on_disk"]]

## Ba truy vấn Spark SQL
Các truy vấn và CSV đã được tạo bởi report_all. Cell dưới in nguyên SQL để giải thích khi bảo vệ; không cần chạy lại các job.

In [ ]:
for title, sql in SQL_QUERIES.items():
    print("\n", title, "\n", sql)

Top doanh thu chỉ bao gồm các hồ sơ FY2024 có trong gói phát hành 2024. Không cộng ba tag doanh thu; chọn tag ưu tiên và một fact hợp nhất cho mỗi công ty. Không coi thứ hạng này là đầy đủ toàn thị trường.

In [ ]:
print((REPORTS / "quality_report.md").read_text(encoding="utf-8"))

In [ ]:
spark.stop()